Chatbot using Hugging Face Transformers
This project implements a conversational chatbot using a pre-trained transformer model from Hugging Face.

**Objective**

To build a chatbot that can interact with users using natural language and generate meaningful responses using transformer-based models.

In [ ]:
# Install required libraries (run once)
!pip install transformers torch

In [ ]:
# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.resize_token_embeddings(len(tokenizer))

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
def run_chatbot():

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Store conversation history
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Give short, clear, and accurate answers."
        }
    ]

    while True:
        user_input = input("You: ")

        # Exit condition (allowed)
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! ")
            break

        # Add user input
        chat_history.append({"role": "user", "content": user_input})

        # Convert to model format
        inputs = tokenizer.apply_chat_template(
            chat_history,
            return_tensors="pt",
            add_generation_prompt=True
        )

        # Move input to model device
        input_ids = inputs["input_ids"].to(model.device)

        # Generate response
        outputs = model.generate(
            input_ids,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

        # Decode response
        response = tokenizer.decode(
            outputs[0][input_ids.shape[-1]:],
            skip_special_tokens=True
        )


        response = response.replace("  ", " ")
        response = response.split("\n")[0]
        response = response.split("###")[0]
        response = response.strip()

        if "." in response:
            response = response.split(".")[0] + "."

        # Safety fallback
        if len(response.strip()) < 5:
            response = "I'm here to help! Could you please ask your question again?"

        print("Chatbot:", response)


        chat_history.append({"role": "assistant", "content": response})



run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hello! How can I assist you today?
You: Who created Python?
Chatbot: Python was created by Guido van Rossum and first released in 1991 as an extension to the Smalltalk programming language.
You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and act like humans.
You: Do you know about Machine Learning?
Chatbot: Yes, Machine Learning is a subset of artificial intelligence that involves training algorithms on data to learn patterns and make predictions or decisions without being explicitly programmed.
You:  Thank you
Chatbot: You're welcome! If you have any more questions, feel free to ask.
You: exit
Chatbot: Goodbye! 
